# siann's TPU probe: JAX and Pallas

Run on a TPU (Colab: Runtime, Change runtime type, TPU; Kaggle: Accelerator, TPU), then Run all: it finishes within
5 minutes. TPU runtimes come with JAX, so nothing is installed. siann's L1 kernel in Pallas is checked against
exact search, then exact search through JAX's built-in operations (L2 and inner product by a matrix multiply and
`lax.top_k` or the TPU's `lax.approx_min_k`; L1 by tiles) is timed against the Pallas kernel on 200,000 Gaussian
vectors, every result checked against NumPy; the benchmark stops itself at what is left of the 5 minutes. Copy
the output of the last two steps back.

In [ ]:
# Step 1 of 5: Which TPU (seconds)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 1 of 5: Which TPU, seconds: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
# which TPU this runtime has, and Python's version (TorchTPU needs 3.11 or later)
import os
import sys

print(sys.version)
for var in ("TPU_ACCELERATOR_TYPE", "TPU_NAME", "COLAB_TPU_ADDR"):
    print(var, os.environ.get(var))
print("accelerator devices:", sorted(f for f in os.listdir("/dev") if f.startswith(("accel", "vfio"))))
print(f"step 1 of 5: Which TPU: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 2 of 5, JAX for the TPU", flush=True)

In [ ]:
# Step 2 of 5: JAX for the TPU (about 1 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 2 of 5: JAX for the TPU, about 1 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
# JAX for TPUs (Colab's TPU runtimes come with it; reinstalled only if it cannot see the TPU)
import subprocess
import sys

r = subprocess.run([sys.executable, "-c", "import jax; print(jax.__version__, jax.devices())"], check=False,
                   capture_output=True, text=True)
if "Tpu" not in r.stdout and "TPU" not in r.stdout:
    print("JAX cannot see a TPU:", (r.stdout + r.stderr).strip().splitlines()[-1:] or "?")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[tpu]"], check=False)
    r = subprocess.run([sys.executable, "-c", "import jax; print(jax.__version__, jax.devices())"], check=False,
                       capture_output=True, text=True)
print((r.stdout + r.stderr).strip())
print(f"step 2 of 5: JAX for the TPU: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 3 of 5, Write jax_probe.py", flush=True)

In [ ]:
# Step 3 of 5: Write jax_probe.py (seconds)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 3 of 5: Write jax_probe.py, seconds: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
source = "\"\"\"A probe of exact search on a Google TPU through JAX: built-in operations for every metric, and a custom kernel\nfor L1 in Pallas (JAX's kernel language, which compiles for TPUs and NVIDIA GPUs, and which PyTorch reaches on\nTPUs through TorchTPU).\n\nBuilt-in: L2 and inner product as a matrix multiply followed by lax.top_k, or by lax.approx_min_k (the TPU's own\napproximate top-k, at a recall target of 0.95), at the TPU's default matrix-multiply precision (bfloat16 inputs)\nand at full float32 (HIGHEST); L1 as a broadcast difference summed over the coordinates, a chunk of the base at a\ntime, with the best k kept across chunks (lax.scan).\n\nPallas, L1: a grid of (query block, base block) programs; each computes its tile of L1 distances in the TPU's\non-chip memory and merges it into its query block's best k, kept in the output across the base blocks, by k\npasses of a minimum (ties by index, as siann's other kernels), so no distance matrix ever reaches memory.\n\nEvery result is checked against exact search in NumPy on a sample of queries (recall@k and the share of queries\nwhose neighbors are exactly NumPy's). Memory: the device's bytes in use and its peak so far, where it reports them.\n\n    python jax_probe.py                                   # on a TPU (Colab: Runtime, Change runtime type, TPU)\n    python jax_probe.py --check                           # the Pallas kernel against NumPy, small shapes\n    python jax_probe.py --check --interpret               # the same anywhere, through Pallas's TPU interpreter\n\"\"\"\nimport argparse\nimport functools\nimport time\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom jax import lax\nfrom jax.experimental import pallas as pl\nfrom jax.experimental.pallas import tpu as pltpu\n\nCHUNK = 256                     # queries per matrix multiply (256 x 1,000,000 float32 scores is 1 GB)\nL1_CHUNK = 16_384               # base vectors per step of the built-in L1 scan\nKP = 128                        # the kept best-k, padded to the TPU's 128 lanes\nBIG = 3.0e38                    # \"no candidate\" (finite: inf - inf would be NaN in a padded row)\nPAD = 1.0e30                    # padding rows of the base: never nearer than a real row\nSLOW = 20.0                     # seconds: a call slower than this is not repeated, and its row's larger batches skipped\nPALLAS_BLOCKS = ((8, 512), (16, 512))                                       # --full: the five below\nPALLAS_BLOCKS_FULL = ((8, 512), (8, 1024), (16, 512), (32, 256), (8, 2048))\n\n\nclass Deadline:\n    \"\"\"A hard limit on the whole run: work checks it before each step, and once it has passed, the step is\n    skipped with a line saying so (the results so far stand).\"\"\"\n\n    def __init__(self, seconds: float):\n        self.seconds, self.end = seconds, time.perf_counter() + seconds\n\n    def over(self, what: str) -> bool:\n        if time.perf_counter() < self.end:\n            return False\n        print(f\"  out of time (the {self.seconds:.0f} s budget): {what} skipped\", flush=True)\n        return True\n\n\nDEADLINE = Deadline(float(\"inf\"))\n\n\ndef exact_numpy(xb, xq, k, metric):\n    \"\"\"NumPy's k nearest (int64 indices, ties by index): the reference.\"\"\"\n    out = []\n    sq = (xb.astype(np.float64) ** 2).sum(axis=1) if metric == \"l2\" else None\n    for q in xq:\n        if metric == \"l2\":\n            d = sq - 2 * (xb @ q).astype(np.float64)             # the rank of |x - q|^2, without a copy of xb\n        elif metric == \"ip\":\n            d = -(xb @ q)\n        else:\n            d = np.abs(xb - q).sum(axis=1)\n        part = np.argpartition(d, k)[:k + 1] if len(d) > k + 1 else np.arange(len(d))\n        cand = np.flatnonzero(d <= d[part].max())\n        out.append(cand[np.lexsort((cand, d[cand]))][:k])\n    return np.array(out)\n\n\n# --- Pallas: L1 with the selection fused -------------------------------------------------------------------------\n\ndef _l1_kernel(q_ref, x_ref, v_ref, i_ref, *, k, block_n):\n    j = pl.program_id(1)\n\n    @pl.when(j == 0)\n    def _():\n        v_ref[...] = jnp.full(v_ref.shape, BIG, jnp.float32)\n        i_ref[...] = jnp.full(i_ref.shape, jnp.iinfo(jnp.int32).max, jnp.int32)\n\n    q, x = q_ref[...], x_ref[...]\n    d = jnp.sum(jnp.abs(q[:, None, :] - x[None, :, :]), axis=2)              # (BQ, BN), on chip\n    di = j * block_n + lax.broadcasted_iota(jnp.int32, d.shape, 1)\n    kv, ki = v_ref[...], i_ref[...]                                            # (BQ, KP), the best so far\n    lane = lax.broadcasted_iota(jnp.int32, kv.shape, 1)\n    nv = jnp.full(kv.shape, BIG, jnp.float32)\n    ni = jnp.full(ki.shape, jnp.iinfo(jnp.int32).max, jnp.int32)\n    for p in range(k):                                                         # k passes of a minimum\n        m = jnp.minimum(jnp.min(kv, axis=1, keepdims=True), jnp.min(d, axis=1, keepdims=True))\n        top = jnp.iinfo(jnp.int32).max\n        mi = jnp.minimum(jnp.min(jnp.where(kv == m, ki, top), axis=1, keepdims=True),\n                         jnp.min(jnp.where(d == m, di, top), axis=1, keepdims=True))   # ties: the smaller index\n        nv = jnp.where(lane == p, m, nv)\n        ni = jnp.where(lane == p, mi, ni)\n        kv = jnp.where((kv == m) & (ki == mi), BIG, kv)\n        d = jnp.where((d == m) & (di == mi), BIG, d)\n    v_ref[...] = nv\n    i_ref[...] = ni\n\n\n@functools.partial(jax.jit, static_argnames=(\"k\", \"block_q\", \"block_n\", \"interpret\"))\ndef pallas_l1(x, q, k, block_q=8, block_n=512, interpret=False):\n    \"\"\"Each query's k nearest rows of x by L1 (int32 indices, nearest first, ties by index), through the kernel.\n    x must already be padded to a multiple of block_n (pad_base).\"\"\"\n    nq, d = q.shape\n    n = x.shape[0]\n    mq = -(-nq // block_q) * block_q\n    q = jnp.pad(q, ((0, mq - nq), (0, 0)))\n    grid = (mq // block_q, n // block_n)\n    params = pltpu.CompilerParams(dimension_semantics=(\"parallel\", \"arbitrary\"))\n    _, i = pl.pallas_call(\n        functools.partial(_l1_kernel, k=k, block_n=block_n),\n        grid=grid,\n        in_specs=[pl.BlockSpec((block_q, d), lambda a, b: (a, 0)),\n                  pl.BlockSpec((block_n, d), lambda a, b: (b, 0))],\n        out_specs=[pl.BlockSpec((block_q, KP), lambda a, b: (a, 0)),\n                   pl.BlockSpec((block_q, KP), lambda a, b: (a, 0))],\n        out_shape=[jax.ShapeDtypeStruct((mq, KP), jnp.float32), jax.ShapeDtypeStruct((mq, KP), jnp.int32)],\n        compiler_params=params,\n        interpret=pltpu.InterpretParams() if interpret else False,\n    )(q, x)\n    return i[:nq, :k]\n\n\ndef pad_base(x, block_n):\n    n = x.shape[0]\n    m = -(-n // block_n) * block_n\n    return jnp.pad(x, ((0, m - n), (0, 0)), constant_values=PAD)\n\n\n# --- built-in operations -----------------------------------------------------------------------------------------\n\ndef builtin_searches(x, k, full=False):\n    sq = jnp.sum(x * x, axis=1)\n\n    def scores(q, metric, precision):\n        s = jnp.matmul(q, x.T, precision=precision)\n        return sq[None, :] - 2 * s if metric == \"l2\" else -s\n\n    def exact(q, metric, precision):\n        return lax.top_k(-scores(q, metric, precision), k)[1]\n\n    def approx(q, metric, precision):\n        return lax.approx_min_k(scores(q, metric, precision), k, recall_target=0.95)[1]\n\n    precisions = {\"default\": lax.Precision.DEFAULT, \"highest\": lax.Precision.HIGHEST}\n    if full:\n        plan = [(m, how, p) for m in (\"l2\", \"ip\") for p in precisions for how in (\"top_k\", \"approx_min_k\")]\n    else:                                   # the short default: what differs, once each\n        plan = [(\"l2\", \"top_k\", \"default\"), (\"l2\", \"top_k\", \"highest\"), (\"l2\", \"approx_min_k\", \"default\"),\n                (\"ip\", \"top_k\", \"default\")]\n    out = {}\n    for metric, how, pname in plan:\n        fn = exact if how == \"top_k\" else approx\n        out[(metric, f\"matmul + {how}, precision {pname}\")] = jax.jit(\n            functools.partial(fn, metric=metric, precision=precisions[pname]))\n    return out\n\n\ndef builtin_l1(x, k):\n    n, d = x.shape\n    m = -(-n // L1_CHUNK) * L1_CHUNK\n    xs = jnp.pad(x, ((0, m - n), (0, 0)), constant_values=PAD).reshape(m // L1_CHUNK, L1_CHUNK, d)\n\n    @jax.jit\n    def search(q):\n        def step(carry, t):\n            bv, bi = carry\n            xc, base = t\n            dist = jnp.sum(jnp.abs(q[:, None, :] - xc[None, :, :]), axis=2)\n            v, i = lax.top_k(-dist, k)\n            v, i = jnp.concatenate([bv, -v], 1), jnp.concatenate([bi, i + base], 1)\n            top = lax.top_k(-v, k)[1]\n            return (jnp.take_along_axis(v, top, 1), jnp.take_along_axis(i, top, 1)), None\n        init = (jnp.full((q.shape[0], k), BIG, jnp.float32), jnp.zeros((q.shape[0], k), jnp.int32))\n        (_, bi), _ = lax.scan(step, init, (xs, jnp.arange(m // L1_CHUNK, dtype=jnp.int32) * L1_CHUNK))\n        return bi\n    return search\n\n\ndef run_chunks(fn, q, size):\n    \"\"\"fn over q in chunks of one fixed size (the last one padded), so each jitted fn compiles once.\"\"\"\n    out = []\n    for s in range(0, q.shape[0], size):\n        qs = q[s:s + size]\n        if qs.shape[0] < size:\n            qs = jnp.concatenate([qs, jnp.broadcast_to(qs[:1], (size - qs.shape[0], qs.shape[1]))])\n        out.append(fn(qs)[:min(size, q.shape[0] - s)])\n    return jnp.concatenate(out)\n\n\ndef memory():\n    \"\"\"The device's bytes in use and its peak so far, MB (None where it does not say).\"\"\"\n    try:\n        st = jax.devices()[0].memory_stats() or {}\n    except Exception:  # noqa: BLE001\n        st = {}\n    get = st.get\n    return tuple(None if get(key) is None else get(key) / 2**20 for key in (\"bytes_in_use\", \"peak_bytes_in_use\"))\n\n\ndef check(interpret: bool) -> None:\n    rng = np.random.default_rng(1)\n    print(f\"  on {jax.devices()[0].platform}{', through the TPU interpreter' if interpret else ''}\", flush=True)\n    for n, d, nq, k, bq, bn in ((1000, 128, 37, 10, 8, 128), (777, 128, 5, 3, 8, 256), (2048, 128, 20, 10, 16, 512)):\n        x = rng.integers(0, 4, (n, d)).astype(np.float32)              # small integers: many ties\n        q = rng.integers(0, 4, (nq, d)).astype(np.float32)\n        truth = exact_numpy(x, q, k, \"l1\")\n        got = np.asarray(pallas_l1(pad_base(jnp.asarray(x), bn), jnp.asarray(q), k, bq, bn, interpret))\n        same = np.array_equal(got, truth)\n        print(f\"  n {n}, d {d}, {nq} queries, k {k}, blocks {bq} x {bn}: identical to exact search, ties \"\n              f\"included: {same}\", flush=True)\n        assert same\n\n\ndef bench(n: int, d: int, batches, l1_batch: int, check_queries: int, full: bool = False) -> None:\n    k = 10\n    dev = jax.devices()[0]\n    rng = np.random.default_rng(0)\n    xb = rng.standard_normal((n, d), dtype=np.float32)\n    xq = rng.standard_normal((max(batches), d), dtype=np.float32)\n    print(f\"JAX {jax.__version__} on {dev.device_kind} ({dev.platform}): {n:,} Gaussian vectors, {d} dimensions, \"\n          f\"k = {k}\", flush=True)\n    t = time.perf_counter()\n    truth = {m: exact_numpy(xb, xq[:check_queries], k, m) for m in (\"l2\", \"ip\")}\n    truth[\"l1\"] = exact_numpy(xb, xq[:min(check_queries, l1_batch)], k, \"l1\")\n    print(f\"  NumPy's exact neighbors for the check: {time.perf_counter() - t:.1f} s\", flush=True)\n    x, q = jnp.asarray(xb), jnp.asarray(xq)\n\n    start = time.perf_counter()\n\n    def elapsed():\n        return f\"{(time.perf_counter() - start) / 60:.1f} min so far\"\n\n    def timed(call, qb):\n        \"\"\"Queries per second, best of two after a warm-up; a call past SLOW s is timed by its warm-up alone and\n        flagged slow (its row's larger batches are skipped).\"\"\"\n        t = time.perf_counter()\n        call(qb).block_until_ready()                                   # compiles, warms up\n        warm = time.perf_counter() - t\n        if warm > SLOW:\n            return qb.shape[0] / warm, True\n        best = warm\n        for _ in range(2):\n            t = time.perf_counter()\n            call(qb).block_until_ready()\n            best = min(best, time.perf_counter() - t)\n        return qb.shape[0] / best, False\n\n    def agreement(found, ref):\n        found = np.asarray(found)[:len(ref)]\n        recall = np.mean([len(set(f) & set(r)) / len(r) for f, r in zip(found, ref, strict=True)])\n        return recall, np.mean([np.array_equal(f, r) for f, r in zip(found, ref, strict=True)])\n\n    def mem():\n        used, peak = memory()\n        return f\"{'?' if used is None else f'{used:,.0f}'} / {'?' if peak is None else f'{peak:,.0f}'}\"\n\n    searches = builtin_searches(x, k, full)\n    blocks = PALLAS_BLOCKS_FULL if full else PALLAS_BLOCKS\n    rows = len(searches) + 1 + len(blocks)\n    print(f\"  queries per second; then the device's memory after the row, MB: in use / peak so far; then recall@10 \"\n          f\"and exact matches against NumPy. {rows} rows; '-' where a batch was skipped (a call past {SLOW:.0f} s \"\n          f\"skips its row's larger batches)\", flush=True)\n    print(f\"  {'':50s}\" + \"\".join(f\"{f'batch {b:,}':>14s}\" for b in batches), flush=True)\n    row = 0\n    for (metric, how), fn in searches.items():\n        def call(qb, fn=fn):\n            return run_chunks(fn, qb, CHUNK)\n        row += 1\n        if DEADLINE.over(f\"row {row} of {rows} and later ones\"):\n            return\n        print(f\"  {f'[{row}/{rows}] {metric}, {how}':50s}\", end=\"\", flush=True)\n        try:\n            slow = False\n            for b in batches:\n                if slow:\n                    print(f\"{'-':>14s}\", end=\"\", flush=True)\n                    continue\n                qps, slow = timed(call, q[:b])\n                print(f\"{qps:14,.0f}\", end=\"\", flush=True)\n            recall, same = agreement(call(q[:check_queries]), truth[metric])\n            print(f\"   ({elapsed()})\\n      memory {mem()}; recall@10 {recall:.4f}, exact matches {same:.4f}\",\n                  flush=True)\n        except Exception as e:  # noqa: BLE001 (report and go on: one failure should not hide the rest)\n            print(f\"\\n      failed: {type(e).__name__}: {str(e)[:300]}\", flush=True)\n\n    qb = q[:l1_batch]\n    print(f\"\\n  L1, batch {l1_batch} (queries per second)\", flush=True)\n    search = builtin_l1(x, k)\n    row += 1\n    if DEADLINE.over(f\"rows {row}-{rows}\"):\n        return\n    try:\n        def call_builtin(qq):\n            return run_chunks(search, qq, 16)\n        qps, _ = timed(call_builtin, qb)\n        recall, same = agreement(call_builtin(qb), truth[\"l1\"])\n        print(f\"    [{row}/{rows}] built-in tiles (lax.scan){qps:18,.0f}   memory {mem()}; recall@10 {recall:.4f}, \"\n              f\"exact matches {same:.4f}; {elapsed()}\", flush=True)\n    except Exception as e:  # noqa: BLE001\n        print(f\"    built-in tiles failed: {type(e).__name__}: {str(e)[:300]}\", flush=True)\n    for bq, bn in blocks:\n        row += 1\n        if DEADLINE.over(f\"rows {row}-{rows}\"):\n            return\n        try:\n            xp = pad_base(x, bn)\n\n            def call_pallas(qq, xp=xp, bq=bq, bn=bn):\n                return pallas_l1(xp, qq, k, bq, bn)\n            qps, _ = timed(call_pallas, qb)\n            recall, same = agreement(call_pallas(qb), truth[\"l1\"])\n            print(f\"    [{row}/{rows}] Pallas, blocks {bq:>2} x {bn:<5}{qps:18,.0f}   memory {mem()}; recall@10 \"\n                  f\"{recall:.4f}, exact matches {same:.4f}; {elapsed()}\", flush=True)\n        except Exception as e:  # noqa: BLE001\n            print(f\"    [{row}/{rows}] Pallas, blocks {bq} x {bn} failed: {type(e).__name__}: {str(e)[:300]}\", flush=True)\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--check\", action=\"store_true\")\n    p.add_argument(\"--interpret\", action=\"store_true\", help=\"with --check: through Pallas's TPU interpreter\")\n    p.add_argument(\"--n\", type=int, default=200_000)\n    p.add_argument(\"--budget\", type=float, default=150, help=\"seconds for the whole run; past it, the rest is skipped\")\n    p.add_argument(\"--d\", type=int, default=128)\n    p.add_argument(\"--batches\", default=\"100,1000\")\n    p.add_argument(\"--l1-batch\", type=int, default=100)\n    p.add_argument(\"--check-queries\", type=int, default=20)\n    p.add_argument(\"--full\", action=\"store_true\", help=\"every search, precision and Pallas block shape\")\n    a = p.parse_args()\n    global DEADLINE\n    DEADLINE = Deadline(a.budget)\n    if a.check:\n        check(a.interpret)\n    else:\n        bench(a.n, a.d, tuple(map(int, a.batches.split(\",\"))), a.l1_batch, a.check_queries, a.full)\n\n\nif __name__ == \"__main__\":\n    main()\n"
with open('jax_probe.py', 'w', encoding='utf-8') as f:
    f.write(source)
print(f"wrote jax_probe.py: {len(source.splitlines())} lines")
print(f"step 3 of 5: Write jax_probe.py: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 4 of 5, Check the Pallas kernel against exact search", flush=True)

In [ ]:
# Step 4 of 5: Check the Pallas kernel against exact search (about 1 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 4 of 5: Check the Pallas kernel against exact search, about 1 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
!python -u jax_probe.py --check
print(f"step 4 of 5: Check the Pallas kernel against exact search: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 5 of 5, JAX benchmark", flush=True)

In [ ]:
# Step 5 of 5: JAX benchmark (about 3 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 5 of 5: JAX benchmark, about 3 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
!python -u jax_probe.py --budget {max(20, int(270 - (__import__('time').time() - NB_START)))}
print(f"step 5 of 5: JAX benchmark: done in {(__import__('time').time() - _t0) / 60:.1f} min; that was the last step", flush=True)